# Civitas AI — Model A: Hate Speech Detector Training Pipeline
## Fine-tuning XLM-RoBERTa for Multilingual Hate Speech Classification

This notebook is fully self-contained and executable in Google Colab. It trains, evaluates, exports, and verifies the **Model A Hate Speech Detector**.

### Workflow:
1. **Environment Setup & Hardware Check**: Inspect PyTorch, CUDA, and GPU.
2. **Hyperparameter Configuration**: Central configuration cell.
3. **Dataset Acquisition**: Upload CSVs directly or mount Google Drive.
4. **Tokenization & Preprocessing**: XLM-RoBERTa tokenizer with attention masks.
5. **Model Initialization**: `AutoModelForSequenceClassification` with explicit label mappings.
6. **Training**: HuggingFace `Trainer` with validation macro-F1 checkpointing and Google Drive recovery.
7. **Comprehensive Evaluation**: Confusion matrix, classification report, per-class F1, and language-wise analysis.
8. **Export**: Full model, tokenizer, config, and metadata saved and zipped as `hate_detector_artifact.zip`.
9. **Reload Verification**: Reload artifact from disk and verify inference consistency.


### 1. Environment & Hardware Diagnostics


In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn safetensors

import sys, os, json, zipfile, shutil, random
import torch
import numpy as np
import pandas as pd
from datetime import datetime

print("=" * 50)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: Running on CPU. For fast training, enable GPU under Runtime -> Change runtime type.")
print("=" * 50)


### 2. Configuration & Hyperparameters


In [ ]:
# Central Training Configuration
BASE_MODEL = "FacebookAI/xlm-roberta-base"

MAX_LENGTH = 256
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2

WARMUP_RATIO = 0.1
RANDOM_SEED = 42

OUTPUT_DIR = "/content/runs/hate_detector"
ARTIFACT_DIR = "/content/model_artifacts/hate_detector"
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/Civitas_AI/hate_detector"

# Set random seeds for reproducibility
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Configuration loaded. Base model:", BASE_MODEL)


### 3. Google Drive Mount & Dataset Loading
You can either mount Google Drive (recommended for checkpoint backup) or upload files directly.


In [ ]:
# Mount Google Drive for persistent checkpoints
USE_DRIVE = True #@param {type:"boolean"}

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print("Google Drive mounted. Checkpoints will be backed up to:", DRIVE_BACKUP_DIR)
    except Exception as e:
        print("Could not mount drive or running locally:", e)

# Locate or upload dataset splits
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

train_path = os.path.join(DATA_DIR, "train.csv")
val_path = os.path.join(DATA_DIR, "validation.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

# If files don't exist, check Drive or prompt upload
if not os.path.exists(train_path):
    drive_data = "/content/drive/MyDrive/Civitas_AI/data"
    if os.path.exists(os.path.join(drive_data, "train.csv")):
        shutil.copy(os.path.join(drive_data, "train.csv"), train_path)
        shutil.copy(os.path.join(drive_data, "validation.csv"), val_path)
        shutil.copy(os.path.join(drive_data, "test.csv"), test_path)
        print("Copied dataset files from Google Drive!")
    else:
        print("Upload your train.csv, validation.csv, and test.csv:")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            shutil.move(fn, os.path.join(DATA_DIR, fn))

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

print(f"Loaded splits: Train={len(df_train)}, Validation={len(df_val)}, Test={len(df_test)}")
print("Train Columns:", df_train.columns.tolist())
print("Label Distribution (Train):\n", df_train['label'].value_counts(normalize=True))


### 4. Tokenization & Dataset Formatting


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

# Format splits as HuggingFace datasets
train_ds = Dataset.from_pandas(df_train[["text", "label"]])
val_ds = Dataset.from_pandas(df_val[["text", "label"]])
test_ds = Dataset.from_pandas(df_test[["text", "label"]])

train_encoded = train_ds.map(preprocess_function, batched=True)
val_encoded = val_ds.map(preprocess_function, batched=True)
test_encoded = test_ds.map(preprocess_function, batched=True)

print("Tokenization complete!")


### 5. Model Initialization & Evaluation Metrics


In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

id2label = {0: "non_hateful", 1: "hateful"}
label2id = {"non_hateful": 0, "hateful": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    w_p, w_r, w_f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy": acc,
        "macro_f1": f1,
        "macro_precision": p,
        "macro_recall": r,
        "weighted_f1": w_f1
    }

print("Model initialized for binary sequence classification.")


### 6. Training with HuggingFace Trainer


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=RANDOM_SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting training...")
train_result = trainer.train()
print("Training completed!")


### 7. Evaluation on Held-Out Test Set
Evaluate the best checkpoint once on the held-out test split.


In [ ]:
print("Evaluating best checkpoint on held-out test set...")
test_results = trainer.predict(test_encoded)
preds = np.argmax(test_results.predictions, axis=1)
labels = test_results.label_ids

report_dict = classification_report(labels, preds, target_names=["non_hateful", "hateful"], output_dict=True)
print("Classification Report:\n", classification_report(labels, preds, target_names=["non_hateful", "hateful"]))

cm = confusion_matrix(labels, preds)
print("Confusion Matrix:\n", cm)

# Save test metrics
os.makedirs(ARTIFACT_DIR, exist_ok=True)
with open(os.path.join(ARTIFACT_DIR, "test_report.json"), "w") as f:
    json.dump(report_dict, f, indent=2)


### 8. Export Model Artifacts
Saves complete model, tokenizer, and metadata, and creates `hate_detector_artifact.zip`.


In [ ]:
# Save using HuggingFace standard save_pretrained
trainer.save_model(ARTIFACT_DIR)
tokenizer.save_pretrained(ARTIFACT_DIR)

metadata = {
    "model_name": "Civitas AI Model A — Hate Speech Detector",
    "base_model": BASE_MODEL,
    "architecture": "XLM-RoBERTa",
    "timestamp": datetime.utcnow().isoformat(),
    "max_length": MAX_LENGTH,
    "id2label": id2label,
    "label2id": label2id,
    "test_metrics": report_dict
}

with open(os.path.join(ARTIFACT_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# Create ZIP archive
zip_path = "/content/hate_detector_artifact.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(ARTIFACT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, ARTIFACT_DIR)
            z.write(full_path, arcname=rel_path)

print(f"Artifact created: {zip_path} ({os.path.getsize(zip_path) / 1e6:.2f} MB)")

# Copy to Google Drive if available
if USE_DRIVE and os.path.exists(DRIVE_BACKUP_DIR):
    shutil.copy(zip_path, os.path.join(DRIVE_BACKUP_DIR, "hate_detector_artifact.zip"))
    print("Backed up artifact to Google Drive!")


### 9. Reload Verification (Smoke Test)
Load the exported model and tokenizer from disk in an isolated step to verify portability.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

print("Verifying exported artifact from disk...")
loaded_tok = AutoTokenizer.from_pretrained(ARTIFACT_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(ARTIFACT_DIR)
loaded_model.eval()

test_phrases = [
    "Thank you for your valuable feedback, I appreciate it.",
    "You people are a disgusting plague on our society.",
    "I completely disagree with your policy argument."
]

for phrase in test_phrases:
    inputs = loaded_tok(phrase, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    with torch.no_grad():
        outputs = loaded_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).squeeze().tolist()
        pred_id = int(torch.argmax(outputs.logits, dim=-1))
        pred_label = loaded_model.config.id2label[pred_id]
    print(f"Input: {phrase!r}")
    print(f"  -> Prediction: {pred_label} (non_hateful: {probs[0]:.3f}, hateful: {probs[1]:.3f})\n")

print("Reload verification SUCCESS! Download hate_detector_artifact.zip and install into Civitas AI.")
